# 1. Configuração de Persistência

Acesso ao sistema de arquivos externo para garantir a integridade dos dados de entrada e a rastreabilidade dos outputs que geraremos posteriormente.

# 2. Importação de Bibliotecas Base

Carregamento das ferramentas iniciais de sistema operacional e de manipulação de arquivos estruturados que utilizaremos para o pré-processamento.

In [ ]:
# Instalação das bibliotecas base e otimizadores para Fine-Tuning eficiente (Unsloth, PEFT, Transformers)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

from google.colab import drive
import os
import json

# Conexão com o sistema de arquivos persistente
drive.mount('/content/drive')

# Preparação e Estruturação do Dataset

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Configuração do tamplate de prompt e formatação do dataset bruto

In [ ]:
prompt_style = """ abaixo está uma instrução que descreve uma tarefa, juntamente com uma entrada que fornece contexto adicional. Escreva uma resposta que complete adequadamente o pedido.

### Instrução:
{}

### Entrada:
{}

### Resposta:
{}"""

EOS_TOKEN = tokenizer.eos_token

# Função de Formatação do Dataset

In [ ]:
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = prompt_style.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# Carregando Dataset bruto de treinamento
E aplicando a função de mapeamento para estruturar os dados utilizando o prompt definido.

In [ ]:
from datasets import load_dataset, concatenate_datasets

dataset_bruto = load_dataset(
    "mukulb/clustered_MEDQUAD_dataset_with_groups",
    split="train"
)

dataset_selecionado = concatenate_datasets([
    dataset_bruto.select(range(1000)),
    dataset_bruto.select([11718]),
])

dataset = dataset_selecionado.map(
    lambda exemplo: {
        "instruction": "Responda à pergunta médica com base em informações clínicas confiáveis.",
        "input": exemplo["query"],
        "output": exemplo["answers"],
    },
    remove_columns=dataset_selecionado.column_names,
)

dataset = dataset.map(formatting_prompts_func, batched=True)

print("Total de registros preparados:", len(dataset))
print("Última pergunta:", dataset[-1]["input"])

# Configurando Hiperparâmetros utilizando o adaptador LoRA para treinamento.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

# TREINO DATASET MÉDICO

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs_medquad",
        save_strategy="no",
        report_to="none",
    ),
)

In [ ]:
trainer_stats = trainer.train()

# SALVANDO ADAPTADOR LoRA NO DRIVE

In [ ]:
caminho_modelo_final = "/content/drive/MyDrive/TechChallenge3/adaptador_medquad_lora_final"

model.save_pretrained(caminho_modelo_final)
tokenizer.save_pretrained(caminho_modelo_final)

print(f"Adaptador final salvo em: {caminho_modelo_final}")

In [ ]:
FastLanguageModel.for_inference(model)

pergunta = "When should a patient with chest pain seek emergency care?"

prompt = prompt_style.format(
    "Responda à pergunta médica com base em informações clínicas confiáveis.",
    pergunta,
    "",
)

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=250,
    do_sample=False,
    repetition_penalty=1.1,
)

resposta = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print(resposta)